# Lab 10: Fashion MNIST

---
author: Trishla Nair
date: December 6, 2024
embed-resources: true
---

## Introduction

For this lab, we use unsupervised learning to classify images of clothes.

## Methods

In [28]:
# imports
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

### Data

In [26]:
# load data
# download training data from open datasets
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# download test data from open datasets
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

The dataset as 60,000 images where the images are arranged in a 28x28 pattern.

In [3]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

In [ ]:
# summary statistics


In [ ]:
# visualizations


### Models

In [29]:
# get cpu, gpu or mps device for training
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using {device} device for training!")

Using cuda device for training!


In [43]:
#process data for ML
batch_size = 10

# create train data loader
train_dataloader = DataLoader(
    training_data,
    batch_size=batch_size,
)

# create test data loader
test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
)

# check data shapes
for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


# send model to compute device
model = NeuralNetwork().to(device)

# check model structure
print(model)

Shape of X [N, C, H, W]: torch.Size([10, 1, 28, 28])
Shape of y: torch.Size([10]) torch.int64
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [44]:
# define loss function
loss_fn = nn.CrossEntropyLoss()

# define optimizer, try different learning rates!!
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
)

In [45]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [46]:
# train models
epochs = 20
for t in range(epochs):
  print(f"Epoch {t+1}\n-------------------------------")
  train(train_dataloader, model, loss_fn, optimizer)
  test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.292477  [   10/60000]
loss: 2.160408  [ 1010/60000]
loss: 1.771071  [ 2010/60000]
loss: 1.501124  [ 3010/60000]
loss: 1.117721  [ 4010/60000]
loss: 1.214576  [ 5010/60000]
loss: 1.369131  [ 6010/60000]
loss: 0.804302  [ 7010/60000]
loss: 0.942992  [ 8010/60000]
loss: 0.999397  [ 9010/60000]
loss: 0.617344  [10010/60000]
loss: 0.759376  [11010/60000]
loss: 1.026922  [12010/60000]
loss: 0.699660  [13010/60000]
loss: 0.482481  [14010/60000]
loss: 0.516165  [15010/60000]
loss: 1.330591  [16010/60000]
loss: 0.297971  [17010/60000]
loss: 1.536830  [18010/60000]
loss: 0.786273  [19010/60000]
loss: 0.398164  [20010/60000]
loss: 0.163487  [21010/60000]
loss: 0.367945  [22010/60000]
loss: 0.682279  [23010/60000]
loss: 0.694526  [24010/60000]
loss: 0.467480  [25010/60000]
loss: 0.426172  [26010/60000]
loss: 0.130817  [27010/60000]
loss: 1.013061  [28010/60000]
loss: 0.718198  [29010/60000]
loss: 0.478387  [30010/60000]
loss: 0.174278  [31010/60000]


## Results

In [47]:
model_scripted = torch.jit.script(model)

import os

# Create the directory if it doesn't exist
directory = "C:/Users/Trish/Documents/CS 307"
os.makedirs(directory, exist_ok=True)

# Specify the file path
file_path = os.path.join(directory, "fashion-mnist.pt")

# Save the model
model_scripted.save(file_path)

print(f"Model saved to {file_path}")

# Download the file
from google.colab import files
files.download(file_path)

Model saved to C:/Users/Trish/Documents/CS 307/fashion-mnist.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

I built a neural network to classify and it was optimised by SGD

## Discussion

Decreasing the Batch size and learning rate seemed to not help much. However it seems with abatch size of 64 and an lr of 0.01, the model's accuracy is close to the threshold.

### Conclusion

This model is not a good fit.